In [ ]:
import os
import json
import numpy as np
from scipy.spatial.transform import Rotation

import Grasping_Face.grasping as gf
import Grasping_Face.visualize_grasping as gfv

In [ ]:
import os
os.listdir('./models_target/models_cad/custom_ind/')

In [ ]:
# Select one CAD mesh (.obj / .ply)
dataset_name = "custom_ind"
name = "pneumaticfitting_GPC1604"
target_mesh_file = f"./models_target/models_cad/{dataset_name}/{name}.obj"
print(target_mesh_file)
print(os.path.exists(target_mesh_file))

In [ ]:
# 1) Mesh -> Patch -> Patch Pair
result = gf.compute_best_patch_pairs(
    mesh_path=target_mesh_file,
    mesh_max_triangles=1500,
    angle_deg=10,
    min_opening=1.0,
    max_opening=140.0,
    angle_tolerance_deg=10,
    top_k=500,
    prefilter_multiplier=10,
    min_prefilter_pool=400,
    max_faces_after_split=25000,
)

print(f"num_patches={result['num_patches']}, num_candidates={result['num_candidates']}")

In [ ]:
# 2) Patch Pair -> Gripper Pad Collision (yaw-aware)
reports = gf.check_gripper_feasibility_faces_with_yaw(
    result,
    yaw_grid_deg=[0, 90],
    max_face_trials_per_pair=100,
    max_feasible_per_pair=10,
    # sort_key =  'epsilon',
)
print(f"feasible_solutions={len(reports)}")

# # Optional: cylinder-based collision variant
# reports = gf.check_gripper_feasibility_faces_with_rotation(
#     result,
#     max_face_trials_per_pair=40,
#     max_feasible_per_pair=4,
# )
# print(f"feasible_solutions={len(reports)}")


In [ ]:
# Visualization examples (v2 supports limiting display count)
fig0 = gfv.visualize_merged_patches_plotly(result, show=False)
fig0.show(renderer="browser")

In [ ]:
fig1 = gfv.visualize_pairs_centroid_lines(result, max_pairs_show=200, show=False)
fig1.show(renderer="browser")

In [ ]:
fig2 = gfv.visualize_feasible_pairs_pads(result, reports, max_reports_show=250, show=False)
fig2.show(renderer="browser")

# fig2 = gfv.visualize_feasible_pairs_with_cylinder(result, reports, max_reports_show=250, vis_cyl=True, show=False)
# fig2.show(renderer="browser")

In [ ]:
# 3) Build grasp pose and EE delta pose for one feasible solution
if len(reports) == 0:
    raise RuntimeError("No feasible report found. Try another CAD or parameter set.")

r0 = reports[0]
mesh_patches = result["mesh_patches"]
patches = {p["id"]: p for p in result["patches"]}

pi = patches[r0["patch_i"]]
pj = patches[r0["patch_j"]]
fi = r0["face_i"]
fj = r0["face_j"]
ci = mesh_patches.vertices[mesh_patches.faces[fi]].mean(axis=0)
cj = mesh_patches.vertices[mesh_patches.faces[fj]].mean(axis=0)
ni = gf.unit(np.asarray(pi["normal"], float))
nj = gf.unit(np.asarray(pj["normal"], float))
yaw_deg = r0.get("feasible_yaw", 0.0)

# Example object pose from OPE stage (replace with your estimate)
H_OC = np.eye(4)
H_OG, stroke = gf.build_gripper_pose_obj_OPE(
    {"centroid": ci, "normal": ni},
    {"centroid": cj, "normal": nj},
    yaw_deg=yaw_deg,
    H_OC=H_OC,
)
H_EdEn, H_OdEn = gf.ee_delta_pose_des(H_OC, H_OG)

quat = Rotation.from_matrix(H_EdEn[:3, :3]).as_quat()
trans_m = H_EdEn[:3, 3] / 1000.0
pose_quat = np.concatenate([trans_m, quat])

print("stroke(mm)=", stroke)
print("pose_quat[x,y,z,qx,qy,qz,qw]=", pose_quat)

In [ ]:
# Optional: visualize coordinate frames
H_dict = {
    "EE origin": np.eye(4),
    "grasp": H_EdEn,
}
fig_frame = gfv.visualize_frames(H_dict, scale=100, H_OdEn=H_OdEn, result=result)
fig_frame.show(renderer="browser")

## Batch Run

In [ ]:
# Batch run: custom + BOP datasets, save HTML results to ./Grasping_Results
models_root = "./models_target/models_cad"
save_root = "./Grasping_Results"
os.makedirs(save_root, exist_ok=True)

dataset_names = [
    "custom_hh", "custom_ind", "custom",
    "BOP_ITODD", "BOP_XYZIBD", "BOP_IPD", "BOP_TLESS",
]
dataset_names = [d for d in dataset_names if os.path.isdir(os.path.join(models_root, d))]
print("Target datasets:", dataset_names)

for dataset_name in dataset_names:
    dataset_dir = os.path.join(models_root, dataset_name)
    out_dir = os.path.join(save_root, dataset_name)
    os.makedirs(out_dir, exist_ok=True)

    cad_files = sorted([
        f for f in os.listdir(dataset_dir)
        if f.lower().endswith((".obj", ".ply"))
    ])
    print(f"\n[{dataset_name}] CAD files: {len(cad_files)}")

    for cad in cad_files:
        name, _ = os.path.splitext(cad)
        mesh_path = os.path.join(dataset_dir, cad)

        try:
            result = gf.compute_best_patch_pairs(
                mesh_path=mesh_path,
                mesh_max_triangles=1500,
                angle_deg=10,
                min_opening=1.0,
                max_opening=140.0,
                angle_tolerance_deg=10,
                top_k=500,
                prefilter_multiplier=10,
                min_prefilter_pool=400,
                max_faces_after_split=25000,
            )

            reports = gf.check_gripper_feasibility_faces_with_yaw(
                result,
                yaw_grid_deg=[0, 90],
                max_face_trials_per_pair=100,
                max_feasible_per_pair=10,
                # sort_key='epsilon',
            )

            feasible_p = result["top_k"]
            feasible_r = [r for r in reports if r.get("feasible")]
            print(f"{dataset_name} - {name}: patch pairs {len(feasible_p)}, feasible sol. {len(feasible_r)}")

            fig = gfv.visualize_merged_patches_plotly(result)
            fig.write_html(os.path.join(out_dir, f"3-patch_normal_{name}.html"), include_plotlyjs="cdn", full_html=True)

            try:
                fig = gfv.visualize_pairs_centroid_lines(result, max_pairs_show=200)
                fig.write_html(os.path.join(out_dir, f"2-patch_pairs_{name}.html"), include_plotlyjs="cdn", full_html=True)
            except Exception as e:
                print(f"  cannot get patch pairs in {dataset_name}:{name} ({e})")

            try:
                fig = gfv.visualize_feasible_pairs_pads(result, reports, max_reports_show=250)
                fig.write_html(
                    os.path.join(out_dir, f"1-grasping_pairs_feasible_{name}_{len(feasible_r)}sol.html"),
                    include_plotlyjs="cdn",
                    full_html=True,
                )
            except Exception as e:
                print(f"  no feasible pairs in {dataset_name}:{name} ({e})")

        except Exception as e:
            print(f"  failed in {dataset_name}:{name} ({e})")